In [1]:
import os

# --- CONFIGURACIÓN DE RUTAS PARA NOTEBOOK ---
# Pega aquí la ruta de tu carpeta 'data'
data_dir = r"\luisf\Documents\trabajo-California-Traffic-Collision\data"
delta_path = os.path.join(data_dir, "delta_lake")

print(f"Buscando Delta Lake en: {delta_path}")

Buscando Delta Lake en: \luisf\Documents\trabajo-California-Traffic-Collision\data\delta_lake


In [14]:
import pandas as pd
import os

# --- 1. RUTA A LA CARPETA QUE YA TIENE LA VERSIÓN 2 ---
# (Asegúrate de que la carpeta 'collisions' esté en esta ruta)
ruta_base = r"C:\Users\luisf\Documents\trabajo-California-Traffic-Collision\data\raw"

# --- 2. LEER TODOS LOS ARCHIVOS PARQUET ---
print("Leyendo archivos de la Versión 2...")

# Delta Lake guarda los datos en varios archivos .parquet, los buscamos todos:
archivos_parquet = [f for f in os.listdir(ruta_base) if f.endswith('.parquet')]

if not archivos_parquet:
    print("Error: No encontré archivos .parquet. Revisa que la ruta sea correcta.")
else:
    # Leemos y concatenamos todo en un solo DataFrame de Pandas
    lista_df = [pd.read_parquet(os.path.join(ruta_base, f)) for f in archivos_parquet]
    df_final = pd.concat(lista_df, ignore_index=True)

    print(f"¡LOGRADO! Se cargaron {len(df_final)} registros exitosamente.")
    
    # --- 3. VISTA PREVIA ---
    print("\nResumen de las primeras filas:")
    print(df_final.head())

Leyendo archivos de la Versión 2...
¡LOGRADO! Se cargaron 400000 registros exitosamente.

Resumen de las primeras filas:
   case_id db_year  jurisdiction officer_id reporting_district chp_shift  \
0  0081715    2021           NaN        NaN                NaN       NaN   
1  0726202    2021           NaN        NaN                NaN       NaN   
2  3858022    2021           NaN        NaN                NaN       NaN   
3  3899441    2021           NaN        NaN                NaN       NaN   
4  3899442    2021           NaN        NaN                NaN       NaN   

  population county_city_location county_location special_condition  ...  \
0        NaN                  NaN             NaN               NaN  ...   
1        NaN                  NaN             NaN               NaN  ...   
2        NaN                  NaN             NaN               NaN  ...   
3        NaN                  NaN             NaN               NaN  ...   
4        NaN                  NaN         

In [8]:
import pandas as pd
import os

# --- 1. ASEGÚRATE QUE ESTA RUTA SEA EXACTA ---
# Si tu carpeta en el disco C se llama diferente, cámbialo aquí.
# Ejemplo: si es "C:\Data\delta_lake", pon esa.
base_path = r"C:\Users\luisf\Documents\trabajo-California-Traffic-Collision\data\delta_lake" 

def cargar_tabla(nombre_carpeta):
    ruta = os.path.join(base_path, nombre_carpeta)
    # Verificamos si la carpeta existe antes de intentar leer
    if not os.path.exists(ruta):
        print(f"❌ ERROR: No existe la carpeta: {ruta}")
        return pd.DataFrame()
    
    # Buscamos archivos .parquet
    archivos = [f for f in os.listdir(ruta) if f.endswith('.parquet')]
    if not archivos:
        print(f"⚠️ ADVERTENCIA: No hay archivos .parquet en: {ruta}")
        return pd.DataFrame()
        
    return pd.concat([pd.read_parquet(os.path.join(ruta, f)) for f in archivos], ignore_index=True)

# Cargamos las tablas
print("Cargando datos...")
collisions = cargar_tabla("collisions")
victims_clean = cargar_tabla("involved_victims")
parties = cargar_tabla("parties")
victims_raw = cargar_tabla("victims")

# --- 2. ESTO DEBE MOSTRAR NÚMEROS MAYORES A 0 ---
if not collisions.empty:
    print(f"✅ ¡ÉXITO! Colisiones cargadas: {len(collisions)} filas")
    print("\nColumnas disponibles en Colisiones:", collisions.columns.tolist())
else:
    print("❌ Las tablas siguen vacías. Revisa la ruta 'base_path' arriba.")

Cargando datos...
✅ ¡ÉXITO! Colisiones cargadas: 600000 filas

Columnas disponibles en Colisiones: ['case_id', 'jurisdiction', 'officer_id', 'reporting_district', 'chp_shift', 'population', 'county_city_location', 'county_location', 'special_condition', 'beat_type', 'chp_beat_type', 'city_division_lapd', 'chp_beat_class', 'beat_number', 'primary_road', 'secondary_road', 'distance', 'direction', 'intersection', 'weather_1', 'weather_2', 'state_highway_indicator', 'caltrans_county', 'caltrans_district', 'state_route', 'route_suffix', 'postmile_prefix', 'postmile', 'location_type', 'ramp_intersection', 'side_of_highway', 'tow_away', 'collision_severity', 'killed_victims', 'injured_victims', 'party_count', 'primary_collision_factor', 'pcf_violation_category', 'pcf_violation', 'pcf_violation_subsection', 'hit_and_run', 'type_of_collision', 'motor_vehicle_involved_with', 'pedestrian_action', 'road_surface', 'road_condition_1', 'road_condition_2', 'lighting', 'control_device', 'chp_road_type'

In [12]:
# SECCIÓN: CONSULTAS DE POBLACIÓN Y RIESGO 

# Q3. Colisiones ocurridas bajo condiciones climáticas adversas (No despejado)
# Filtramos todo lo que NO sea 'clear'
q3 = collisions[collisions['weather_1'] != 'clear']
print(f"Q3: Colisiones bajo condiciones climaticas adversas: {len(q3)} filas")

# Q4. Víctimas que no portaban equipo de seguridad (cinturón, casco, etc.)
# Usamos 'victims_raw' para detalles específicos del equipo
q4 = victims_raw[victims_raw['victim_safety_equipment_1'].str.contains('none', na=False, case=False)]
print(f"Q4: Victimas sin equipo de seguridad: {len(q4)} filas")

# Q5. Accidentes ocurridos en condiciones de oscuridad (con o sin alumbrado)
q5 = collisions[collisions['lighting'].str.contains('dark', na=False, case=False)]
print(f"Q5: Colisiones en condiciones de oscuridad: {len(q5)} filas")

# Q6. Colisiones que tuvieron lugar en intersecciones
q6 = collisions[collisions['location_type'] == 'intersection']
print(f"Q6: Colisiones en intersecciones: {len(q6)} filas")

# Q7. Perfil de riesgo: Mujeres de la tercera edad (> 65 años)
# IMPORTANTE: Usamos 'victims_clean' (tu tabla sin repetidos) para estadística real
q7 = victims_clean[(victims_clean['victim_sex'] == 'female') & (victims_clean['victim_age'] > 65)]
print(f"Q7: Víctimas de la tercera edad: {len(q7)} filas")

# Q8. Accidentes donde estuvieron involucradas motocicletas
q8 = parties[parties['statewide_vehicle_type'] == 'motorcycle']
print(f"Q8: Accidentes con motocicletas: {len(q8)} filas")

# Q9. Colisiones causadas por exceso de velocidad (PCF Violation)
q9 = collisions[collisions['pcf_violation_category'] == 'speeding']
print(f"Q9: Colisiones por exceso de velocidad: {len(q9)} filas")

# Q10. Víctimas que fueron expulsadas del vehículo durante el impacto
q10 = victims_raw[victims_raw['victim_ejected'] == 'yes']
print(f"Q10: Víctimas expulsadas del vehículo: {len(q10)} filas")

Q3: Colisiones bajo condiciones climaticas adversas: 172026 filas
Q4: Victimas sin equipo de seguridad: 5112 filas
Q5: Colisiones en condiciones de oscuridad: 196716 filas
Q6: Colisiones en intersecciones: 11874 filas
Q7: Víctimas de la tercera edad: 3138 filas
Q8: Accidentes con motocicletas: 0 filas
Q9: Colisiones por exceso de velocidad: 187440 filas
Q10: Víctimas expulsadas del vehículo: 0 filas


In [13]:
# SECCIÓN: ANÁLISIS ESTADÍSTICOS ACTUARIALES 

# A1. Letalidad promedio según rangos de edad (Uso de tabla Limpia)
# Cruzamos víctimas con colisiones para obtener el conteo de fallecidos
letalidad_df = pd.merge(victims_clean, collisions[['case_id', 'killed_victims']], on='case_id')
bins_edad = [0, 25, 60, 100]
etiquetas = ['Jóvenes', 'Adultos', 'Ancianos']
print("\nA1. Promedio de fallecidos por rango de edad:")
print(letalidad_df.groupby(pd.cut(letalidad_df['victim_age'], bins=bins_edad, labels=etiquetas))['killed_victims'].mean())

# A2. Relación entre uso de celular y culpabilidad en el accidente
parties['at_fault'] = pd.to_numeric(parties['at_fault'], errors='coerce')
print("\nA2. Probabilidad de ser culpable según uso de celular:")
print(parties.groupby('cellphone_in_use')['at_fault'].mean())

# A3. Top 5 de marcas de vehículos con mayor frecuencia de culpabilidad
print("\nA3. Top 5 marcas de vehículos involucradas en choques por culpa:")
marcas_culpa = parties[parties['at_fault'] == 1]['vehicle_make'].value_counts().head(5)
print(marcas_culpa)

# A4. Severidad del accidente según el estado de la superficie de la vía
print("\nA4. Promedio de heridos según tipo de carretera (Road Surface):")
print(collisions.groupby('road_surface')['injured_victims'].mean())

# A5. Matriz de Sobriedad vs Severidad del Accidente
# Este análisis es fundamental para medir el impacto del alcohol en la gravedad
sobriedad_analisis = pd.merge(parties[['case_id', 'party_sobriety']], 
                              collisions[['case_id', 'collision_severity']], on='case_id')
print("\nA5. Tabla Cruzada: Estado de Sobriedad vs Gravedad del Choque")
print(pd.crosstab(sobriedad_analisis['party_sobriety'], sobriedad_analisis['collision_severity']))


A1. Promedio de fallecidos por rango de edad:


C:\Users\luisf\AppData\Local\Temp\ipykernel_13068\624764559.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(letalidad_df.groupby(pd.cut(letalidad_df['victim_age'], bins=bins_edad, labels=etiquetas))['killed_victims'].mean())


victim_age
Jóvenes     0.012998
Adultos     0.019462
Ancianos    0.026709
Name: killed_victims, dtype: float64

A2. Probabilidad de ser culpable según uso de celular:
cellphone_in_use
0.0    0.477592
1.0    0.536195
Name: at_fault, dtype: float64

A3. Top 5 marcas de vehículos involucradas en choques por culpa:
vehicle_make
toyota       40710
ford         38166
honda        28080
chevrolet    26178
nissan       15570
Name: count, dtype: int64

A4. Promedio de heridos según tipo de carretera (Road Surface):
road_surface
dry         0.539094
slippery    0.571429
snowy       0.444837
wet         0.473858
Name: injured_victims, dtype: float64

A5. Tabla Cruzada: Estado de Sobriedad vs Gravedad del Choque
collision_severity                      fatal  other injury    pain  \
party_sobriety                                                        
had been drinking, impairment unknown     396          3456    4032   
had been drinking, not under influence    612          3708    5724   
had be

In [1]:
import polars as pl

#Ruta a la base de datos 
db_path = "data/California_modificada.db"
uri = f"sqlite://{db_path}"

# 2. Carga de tablas principales (usando connectorx para velocidad)
print("Cargando datos con Polars...")

# Cargamos 'parties' para la sobriedad
df_parties = pl.read_database_uri(query="SELECT case_id, party_sobriety FROM parties", uri=uri, engine="connectorx")

# Cargamos 'collisions' para saber cuántos murieron (killed_victims)
df_collisions = pl.read_database_uri(query="SELECT case_id, killed_victims FROM collisions", uri=uri, engine="connectorx")

# Cargamos 'victims' para el filtro de edad mayor a cero (como mencionaste)
df_victims = pl.read_database_uri(query="SELECT case_id, victim_age FROM victims", uri=uri, engine="connectorx")

print("Datos cargados con éxito")

Cargando datos con Polars...
Datos cargados con éxito


In [2]:
import polars as pl

#RUTA Y CONEXIÓN
path = r"C:\Users\EQUIPO\trabajo-California-Traffic-Collision\data\California_modificada.db"
uri = f"sqlite:///{path}"

try:
    #CARGAMOS VÍCTIMAS 
    df_victims = pl.read_database_uri(
        query="SELECT case_id, victim_age, victim_degree_of_injury FROM victims",
        uri=uri, engine="adbc"
    )
    
    #Creamos la variable para que no de NameError
    df_menores = df_victims.filter(pl.col("victim_age") < 15)

    #CARGAMOS COLISIONES 
    
    df_collisions = pl.read_database_uri(
        query="SELECT case_id, killed_victims, injured_victims FROM collisions",
        uri=uri, engine="adbc"
    )

    #EL JOIN
    df_menores_impacto = df_menores.join(df_collisions, on="case_id", how="inner")

    print(f"TODO CARGADO A LA PRIMERA")
    print(f"Total registros: {df_menores_impacto.height}")
    print("Columnas listas:", df_menores_impacto.columns)

except Exception as e:
    print(f" Error crítico: {e}")

TODO CARGADO A LA PRIMERA
Total registros: 122980
Columnas listas: ['case_id', 'victim_age', 'victim_degree_of_injury', 'killed_victims', 'injured_victims']


In [3]:
import polars as pl

path = r"C:\Users\EQUIPO\trabajo-California-Traffic-Collision\data\California_modificada.db"
uri = f"sqlite:///{path}"

menores = df_victims.filter(pl.col("victim_age") < 15)

# He quitado 'county' y 'crash_date'
df_collisions = pl.read_database_uri(
    query="SELECT case_id, killed_victims, injured_victims FROM collisions",
    uri=uri, engine="adbc"
)

# El JOIN (Ahora debería funcionar porque solo usamos case_id)
df_menores_impacto = menores.join(df_collisions, on="case_id", how="inner")

print(f"CONECTADO Filas: {df_menores_impacto.height}")
print("Columnas disponibles ahora:", df_menores_impacto.columns)

CONECTADO Filas: 122980
Columnas disponibles ahora: ['case_id', 'victim_age', 'victim_degree_of_injury', 'killed_victims', 'injured_victims']


In [5]:
#EXPERIMENTO 1: Distribución de Gravedad por Edad
exp_vulnerabilidad = df_menores_impacto.group_by("victim_age").agg(
    pl.col("killed_victims").sum().alias("Muertes_Totales"),
    pl.col("injured_victims").sum().alias("Heridos_Totales"),
    pl.len().alias("Casos_Registrados")
).sort("victim_age")

print("--- RESULTADO EXPERIMENTO: VULNERABILIDAD POR EDAD ---")
print(exp_vulnerabilidad)

--- RESULTADO EXPERIMENTO: VULNERABILIDAD POR EDAD ---
shape: (15, 4)
┌────────────┬─────────────────┬─────────────────┬───────────────────┐
│ victim_age ┆ Muertes_Totales ┆ Heridos_Totales ┆ Casos_Registrados │
│ ---        ┆ ---             ┆ ---             ┆ ---               │
│ i64        ┆ i64             ┆ i64             ┆ u32               │
╞════════════╪═════════════════╪═════════════════╪═══════════════════╡
│ 0          ┆ 59              ┆ 4499            ┆ 3431              │
│ 1          ┆ 117             ┆ 11476           ┆ 8012              │
│ 2          ┆ 113             ┆ 11707           ┆ 8016              │
│ 3          ┆ 93              ┆ 12215           ┆ 8118              │
│ 4          ┆ 118             ┆ 12807           ┆ 8235              │
│ …          ┆ …               ┆ …               ┆ …                 │
│ 10         ┆ 114             ┆ 15330           ┆ 8684              │
│ 11         ┆ 110             ┆ 15862           ┆ 8955              │
│ 12   

In [6]:
# EXPERIMENTO 2: Top 10 Accidentes con más Menores Afectados
casos_criticos = df_menores_impacto.filter(
    (pl.col("killed_victims") > 0) | (pl.col("injured_victims") > 2)
).sort("killed_victims", descending=True)

print("--- RESULTADO EXPERIMENTO: CASOS DE ALTO IMPACTO ---")
print(casos_criticos.head(10))

--- RESULTADO EXPERIMENTO: CASOS DE ALTO IMPACTO ---
shape: (10, 5)
┌──────────┬────────────┬─────────────────────────┬────────────────┬─────────────────┐
│ case_id  ┆ victim_age ┆ victim_degree_of_injury ┆ killed_victims ┆ injured_victims │
│ ---      ┆ ---        ┆ ---                     ┆ ---            ┆ ---             │
│ i64      ┆ i64        ┆ str                     ┆ i64            ┆ i64             │
╞══════════╪════════════╪═════════════════════════╪════════════════╪═════════════════╡
│ 90765562 ┆ 14         ┆ killed                  ┆ 5              ┆ 1               │
│ 90765562 ┆ 11         ┆ killed                  ┆ 5              ┆ 1               │
│ 90978975 ┆ 5          ┆ killed                  ┆ 5              ┆ 0               │
│ 91486401 ┆ 1          ┆ killed                  ┆ 5              ┆ 0               │
│ 91486401 ┆ 7          ┆ killed                  ┆ 5              ┆ 0               │
│ 8465633  ┆ 13         ┆ killed                  ┆ 4         

In [7]:
tablas = pl.read_database_uri(query="SELECT name FROM sqlite_master WHERE type='table'", uri=uri, engine="adbc")
print("Tablas en la DB:", tablas)

try:
    columnas_parties = pl.read_database_uri(query="SELECT * FROM parties LIMIT 0", uri=uri, engine="adbc")
    print("Columnas en PARTIES:", columnas_parties.columns)
except:
    print("No encontré la tabla PARTIES")

Tablas en la DB: shape: (6, 1)
┌─────────────────┐
│ name            │
│ ---             │
│ str             │
╞═════════════════╡
│ case_ids        │
│ collisions      │
│ parties         │
│ sqlite_sequence │
│ victims         │
│ sqlean_define   │
└─────────────────┘
Columnas en PARTIES: ['id_party', 'case_id', 'party_number', 'party_sobriety', 'party_type', 'party_drug_physical', 'movement_preceding_collision', 'at_fault', 'vehicle_make', 'vehicle_year', 'cellphone_in_use', 'statewide_vehicle_type', 'party_safety_equipment_1', 'party_safety_equipment_2']


In [8]:
import polars as pl

# 1. Cargamos TODA la columna de muertos sin filtros raros
query_verificacion = "SELECT killed_victims FROM collisions"
df_muertos = pl.read_database_uri(query=query_verificacion, uri=uri, engine="adbc")

# 2. Hacemos la suma manual en Python, que es más confiable que el SQL de SQLite
total_real = df_muertos["killed_victims"].sum()
total_filas = df_muertos.height

print("--- REPORTE DE MUERTOS TOTALES ---")
print(f"* Total de accidentes registrados: {total_filas}")
print(f"* Total de personas fallecidas: {total_real}")
print("----------------------------------")

--- REPORTE DE MUERTOS TOTALES ---
* Total de accidentes registrados: 1451814
* Total de personas fallecidas: 11922
----------------------------------


In [9]:
print(pl.read_database_uri(query="SELECT party_drug_physical, COUNT(*) as total FROM parties GROUP BY 1", uri=uri, engine="adbc"))

shape: (6, 2)
┌───────────────────────┬─────────┐
│ party_drug_physical   ┆ total   │
│ ---                   ┆ ---     │
│ str                   ┆ i64     │
╞═══════════════════════╪═════════╡
│ null                  ┆ 2301220 │
│ G                     ┆ 314679  │
│ impairment - physical ┆ 4010    │
│ not applicable        ┆ 222355  │
│ sleepy/fatigued       ┆ 19757   │
│ under drug influence  ┆ 16292   │
└───────────────────────┴─────────┘


In [40]:
import pandas as pd
import numpy as np

# 1. CARGA (Aseguramos que 'vic' exista)
col = pd.read_parquet('data/delta_lake/collisions')
part = pd.read_parquet('data/delta_lake/parties')
vic = pd.read_parquet('data/delta_lake/victims') # AQUÍ ESTÁ TU 'vic'

# 2. LIMPIEZA CON EL MÉTODO DE KLEIBER (IQR) para la EDAD
def limpiar_outliers(df, columna):
    Q1 = df[columna].quantile(0.25)
    Q3 = df[columna].quantile(0.75)
    IQR = Q3 - Q1
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR
    return df[(df[columna] >= limite_inferior) & (df[columna] <= limite_superior)]

# Limpiamos las edades en víctimas para la Pirámide
vic_clean = limpiar_outliers(vic[vic['victim_age'] > 0], 'victim_age')

print("✅ Datos cargados y filtrados con IQR.")

✅ Datos cargados y filtrados con IQR.


In [ ]:
import polars as pl

#Carga de datos con nombres de columnas verificados
query_col = "SELECT case_id, killed_victims FROM collisions"
query_par = "SELECT case_id, party_safety_equipment_1 FROM parties"

df_col = pl.read_database_uri(query=query_col, uri=uri, engine="adbc")
df_par = pl.read_database_uri(query=query_par, uri=uri, engine="adbc")

#Join y limpieza de tipos de datos
df_res = df_col.join(df_par, on="case_id", how="inner").with_columns([
    pl.col("party_safety_equipment_1").cast(pl.String).fill_null("No especificado"),
    pl.col("killed_victims").cast(pl.Int64).fill_null(0)
])

#Procesamiento estadístico
analisis = (
    df_res.group_by("party_safety_equipment_1")
    .agg([
        pl.len().alias("Total_Casos"),
        pl.col("killed_victims").sum().alias("Total_Muertos")
    ])
    .with_columns(
        (pl.col("Total_Muertos") / pl.col("Total_Casos")).alias("Indice_Mortalidad")
    )
    .sort("Indice_Mortalidad", descending=True)
)

# 4. INTERPRETACIÓN AUTOMÁTICA EN EL NOTEBOOK
print("================================================================")
print("   EXPERIMENTO #1: IMPACTO DEL EQUIPO DE SEGURIDAD EN LA VIDA")
print("================================================================\n")
print(analisis)

# Extraemos valores clave para el texto
not_used = analisis.filter(pl.col("party_safety_equipment_1").str.contains("not used")).select("Indice_Mortalidad").to_series()[0]
airbag = analisis.filter(pl.col("party_safety_equipment_1").str.contains("air bag deployed")).select("Indice_Mortalidad").to_series()[0]

print(f"\n--- ANÁLISIS DE RESULTADOS ---")
print(f"1. RIESGO CRÍTICO: El índice de mortalidad cuando NO SE USA CINTURÓN ({not_used:.4f})")
print(f"   es casi TRIPLE que cuando el AIRBAG se despliega ({airbag:.4f}).")
print(f"2. CONCLUSIÓN: La falta de equipo de seguridad pasiva es el factor")
print(f"   que más dispara la letalidad en un accidente")

   EXPERIMENTO #1: IMPACTO DEL EQUIPO DE SEGURIDAD EN LA VIDA

shape: (24, 4)
┌─────────────────────────────────┬─────────────┬───────────────┬───────────────────┐
│ party_safety_equipment_1        ┆ Total_Casos ┆ Total_Muertos ┆ Indice_Mortalidad │
│ ---                             ┆ ---         ┆ ---           ┆ ---               │
│ str                             ┆ u32         ┆ i64           ┆ f64               │
╞═════════════════════════════════╪═════════════╪═══════════════╪═══════════════════╡
│ lap/shoulder harness not used   ┆ 316         ┆ 14            ┆ 0.044304          │
│ not required                    ┆ 62624       ┆ 1932          ┆ 0.030851          │
│ driver, motorcycle helmet used  ┆ 625         ┆ 19            ┆ 0.0304            │
│ driver, motorcycle helmet not … ┆ 94          ┆ 2             ┆ 0.021277          │
│ air bag deployed                ┆ 499873      ┆ 7599          ┆ 0.015202          │
│ …                               ┆ …           ┆ …           

In [42]:
import polars as pl

#Carga de datos de la tabla 'collisions'
# olo necesitamos la hora y los fallecidos
query = "SELECT collision_time, killed_victims FROM collisions"
df_time = pl.read_database_uri(query=query, uri=uri, engine="adbc")

#PROCESAMIENTO: Extraemos la hora y limpiamos
# Convertimos el tiempo a la hora (0-23) para poder comparar bloques

df_hourly = df_time.with_columns([
    pl.col("collision_time").cast(pl.String).str.slice(0, 2).alias("hora"),
    pl.col("killed_victims").cast(pl.Int64).fill_null(0)
]).filter(pl.col("hora").str.contains(r"^\d+$")) # Filtramos solo los que son números válidos

#CÁLCULO DE LETALIDAD POR HORA
reporte_hora = (
    df_hourly.group_by("hora")
    .agg([
        pl.len().alias("Total_Choques"),
        pl.col("killed_victims").sum().alias("Total_Muertos")
    ])
    .with_columns(
        (pl.col("Total_Muertos") / pl.col("Total_Choques")).alias("Indice_Letalidad")
    )
    .sort("Indice_Letalidad", descending=True)
)
#INTERPRETACIÓN
print("================================================================")
print("   EXPERIMENTO #2: EL PRIME TIME DE LA FATALIDAD (POR HORA)")
print("================================================================\n")
print(reporte_hora.head(5)) # Mostramos las 5 horas más peligrosas

#Lógica para extraer la peor hora
peor_hora = reporte_hora["hora"][0]
peor_indice = reporte_hora["Indice_Letalidad"][0]

print(f"\n---------- ANÁLISIS DE RESULTADOS-------------")
print(f"PUNTO CRÍTICO: La hora {peor_hora}:00 am presenta el mayor índice")
print(f"   de letalidad con un valor de ({peor_indice:.4f}).")
print(f"----Esto permite identificar si los accidentes fatales")
print(f"   están ligados a la visibilidad (noche) o al volumen de tráfico.")
print(f"----La hora está entre las 00:00 y las 05:00 am, la letalidad")
print(f"   suele asociarse a fatiga o consumo de sustancias.")

   EXPERIMENTO #2: EL PRIME TIME DE LA FATALIDAD (POR HORA)

shape: (5, 4)
┌──────┬───────────────┬───────────────┬──────────────────┐
│ hora ┆ Total_Choques ┆ Total_Muertos ┆ Indice_Letalidad │
│ ---  ┆ ---           ┆ ---           ┆ ---              │
│ str  ┆ u32           ┆ i64           ┆ f64              │
╞══════╪═══════════════╪═══════════════╪══════════════════╡
│ 01   ┆ 28601         ┆ 521           ┆ 0.018216         │
│ 23   ┆ 39139         ┆ 648           ┆ 0.016556         │
│ 04   ┆ 21637         ┆ 358           ┆ 0.016546         │
│ 03   ┆ 21125         ┆ 344           ┆ 0.016284         │
│ 02   ┆ 27255         ┆ 440           ┆ 0.016144         │
└──────┴───────────────┴───────────────┴──────────────────┘

---------- ANÁLISIS DE RESULTADOS-------------
PUNTO CRÍTICO: La hora 01:00 am presenta el mayor índice
   de letalidad con un valor de (0.0182).
----Esto permite identificar si los accidentes fatales
   están ligados a la visibilidad (noche) o al volumen de tráfi

In [48]:
import polars as pl

#Carga de datos de la tabla 'collisions'
query = "SELECT type_of_collision, killed_victims FROM collisions"
df_type = pl.read_database_uri(query=query, uri=uri, engine="adbc")

#PROCESAMIENTO: Limpieza y Casting
df_clean = df_type.with_columns([
    pl.col("type_of_collision").cast(pl.String).fill_null("No especificado"),
    pl.col("killed_victims").cast(pl.Int64).fill_null(0)
])
#CÁLCULO DE LETALIDAD
reporte_tipo = (
    df_clean.group_by("type_of_collision")
    .agg([
        pl.len().alias("Total_Choques"),
        pl.col("killed_victims").sum().alias("Total_Muertos")
    ])
    .with_columns(
        (pl.col("Total_Muertos") / pl.col("Total_Choques")).alias("Indice_Letalidad")
    )
    .sort("Indice_Letalidad", descending=True)
)
#INTERPRETACIÓN
print("================================================================")
print("   EXPERIMENTO #3: LETALIDAD POR TIPO DE COLISIÓN")
print("================================================================\n")
print(reporte_tipo)

#Lógica para extraer el peor tipo
peor_tipo = reporte_tipo["type_of_collision"][0]
indice_peor = reporte_tipo["Indice_Letalidad"][0]

print(f"\n--- ANÁLISIS DE RESULTADOS ---")
print(f"1. CONFIGURACIÓN MÁS PELIGROSA: El tipo '{peor_tipo}' es el más letal")
print(f"   con un índice de ({indice_peor:.4f}).")
print(f"2. OBSERVACIÓN: Los choques frontales ('head-on') suelen liderar")
print(f"   esta lista debido a la energía del impacto doble.")
print(f"3. Si el tipo es 'hit object', indica que las infraestructuras")
print(f"   viales (postes, muros) son trampas mortales a alta velocidad.")
print(f"Un peatón tiene casi 35 veces más probabilidades de morir en un choque que alguien en un carro donde se despliega un airbag (0.002). 35 veces! DIABLAZO")

   EXPERIMENTO #3: LETALIDAD POR TIPO DE COLISIÓN

shape: (12, 4)
┌───────────────────┬───────────────┬───────────────┬──────────────────┐
│ type_of_collision ┆ Total_Choques ┆ Total_Muertos ┆ Indice_Letalidad │
│ ---               ┆ ---           ┆ ---           ┆ ---              │
│ str               ┆ u32           ┆ i64           ┆ f64              │
╞═══════════════════╪═══════════════╪═══════════════╪══════════════════╡
│ pedestrian        ┆ 40028         ┆ 2888          ┆ 0.072149         │
│ head-on           ┆ 66167         ┆ 1519          ┆ 0.022957         │
│ overturned        ┆ 35286         ┆ 788           ┆ 0.022332         │
│ hit object        ┆ 244223        ┆ 2878          ┆ 0.011784         │
│ other             ┆ 36574         ┆ 296           ┆ 0.008093         │
│ …                 ┆ …             ┆ …             ┆ …                │
│ rear end          ┆ 466277        ┆ 1062          ┆ 0.002278         │
│ sideswipe         ┆ 293802        ┆ 491           ┆ 0.00

In [ ]:
import polars as pl

#Carga y limpieza 
query = "SELECT type_of_collision, killed_victims FROM collisions"
df_type = pl.read_database_uri(query=query, uri=uri, engine="adbc")

df_clean = df_type.with_columns([
    pl.col("type_of_collision").cast(pl.String).fill_null("No especificado"),
    pl.col("killed_victims").cast(pl.Int64).fill_null(0)
])

#Procesamiento
reporte_tipo = (
    df_clean.group_by("type_of_collision")
    .agg([
        pl.len().alias("Total_Choques"),
        pl.col("killed_victims").sum().alias("Total_Muertos")
    ])
    .with_columns(
        (pl.col("Total_Muertos") / pl.col("Total_Choques")).alias("Indice_Letalidad")
    )
    .sort("Indice_Letalidad", descending=True)
)
#INTERPRETACIÓN
print("================================================================")
print("   EXPERIMENTO #3: VULNERABILIDAD SEGÚN TIPO DE IMPACTO")
print("================================================================\n")
print(reporte_tipo)

#Extraemos los datos críticos
pedestrian_idx = reporte_tipo.filter(pl.col("type_of_collision") == "pedestrian")["Indice_Letalidad"][0]
head_on_idx = reporte_tipo.filter(pl.col("type_of_collision") == "head-on")["Indice_Letalidad"][0]
rear_end_idx = reporte_tipo.filter(pl.col("type_of_collision") == "rear end")["Indice_Letalidad"][0]

print(f"\n--- ANÁLISIS DE RESULTADOS ---")
print(f"1. EL PEATÓN ES EL ESLABÓN MÁS DÉBIL: Con un índice de {pedestrian_idx:.4f},")
print(f"   ser atropellado es la situación más mortal en California.")
print(f"2. ENERGÍA CINÉTICA: El choque frontal ('head-on') tiene una letalidad de {head_on_idx:.4f},")
print(f"   siendo DIEZ VECES más peligroso que un choque por alcance ('rear end' con {rear_end_idx:.4f}).")
print(f"3. CONCLUSIÓN: La infraestructura debe proteger al peatón; una vez ocurre el")
print(f"   contacto, la probabilidad de muerte se dispara exponencialmente.")

   EXPERIMENTO #3: VULNERABILIDAD SEGÚN TIPO DE IMPACTO

shape: (12, 4)
┌───────────────────┬───────────────┬───────────────┬──────────────────┐
│ type_of_collision ┆ Total_Choques ┆ Total_Muertos ┆ Indice_Letalidad │
│ ---               ┆ ---           ┆ ---           ┆ ---              │
│ str               ┆ u32           ┆ i64           ┆ f64              │
╞═══════════════════╪═══════════════╪═══════════════╪══════════════════╡
│ pedestrian        ┆ 40028         ┆ 2888          ┆ 0.072149         │
│ head-on           ┆ 66167         ┆ 1519          ┆ 0.022957         │
│ overturned        ┆ 35286         ┆ 788           ┆ 0.022332         │
│ hit object        ┆ 244223        ┆ 2878          ┆ 0.011784         │
│ other             ┆ 36574         ┆ 296           ┆ 0.008093         │
│ …                 ┆ …             ┆ …             ┆ …                │
│ rear end          ┆ 466277        ┆ 1062          ┆ 0.002278         │
│ sideswipe         ┆ 293802        ┆ 491           

In [52]:
print("Si no se usa cinturón y sale a la 1:00 AM en California 2018-2021, lo más probable es que terminarías en un 'head-on' o atropellando a alguien, y ahí la posibilidad de muerte es casi del 10%")

Si no se usa cinturón y sale a la 1:00 AM en California 2018-2021, lo más probable es que terminarías en un 'head-on' o atropellando a alguien, y ahí la posibilidad de muerte es casi del 10%


In [56]:
import polars as pl

#Cargamos datos de ambas tablas
query_col = "SELECT case_id, killed_victims FROM collisions"
query_par = "SELECT case_id, vehicle_year FROM parties WHERE party_type = 'driver'"

df_col = pl.read_database_uri(query=query_col, uri=uri, engine="adbc")
df_par = pl.read_database_uri(query=query_par, uri=uri, engine="adbc")

#Join y Limpieza
df_veh = df_col.join(df_par, on="case_id", how="inner").with_columns([
    pl.col("vehicle_year").cast(pl.Int64, strict=False).fill_null(0),
    pl.col("killed_victims").cast(pl.Int64).fill_null(0)
])
#Creamos Rangos de Antigüedad (Corregido con is_between)
df_buckets = df_veh.with_columns(
    pl.when(pl.col("vehicle_year").is_between(1900, 2000)).then(pl.lit("1. Clásicos/Viejos (<2000)"))
    .when(pl.col("vehicle_year").is_between(2001, 2010)).then(pl.lit("2. Década 2000 (2001-2010)"))
    .when(pl.col("vehicle_year").is_between(2011, 2020)).then(pl.lit("3. Modernos (2011-2020)"))
    .when(pl.col("vehicle_year") > 2020).then(pl.lit("4. Último Modelo (>2020)"))
    .otherwise(pl.lit("5. Desconocido/Otros"))
    .alias("era_vehiculo")
)
#Cálculo de Impacto
reporte_vehiculo = (
    df_buckets.group_by("era_vehiculo")
    .agg([
        pl.len().alias("Total_Accidentes"),
        pl.col("killed_victims").sum().alias("Total_Muertos")
    ])
    .with_columns(
        (pl.col("Total_Muertos") / pl.col("Total_Accidentes")).alias("Indice_Mortalidad")
    )
    .sort("era_vehiculo")
)

#INTERPRETACIÓN 
print("================================================================")
print("   EXPERIMENTO #4: SEGURIDAD VEHICULAR (AÑO DEL CARRO VS. MUERTE)")
print("================================================================\n")
print(reporte_vehiculo)

#Lógica para el análisis de los resultados obtenidos
try:
    viejos = reporte_vehiculo.filter(pl.col("era_vehiculo").str.contains("Viejos"))["Indice_Mortalidad"][0]
    modernos = reporte_vehiculo.filter(pl.col("era_vehiculo").str.contains("Modernos"))["Indice_Mortalidad"][0]
    
    print(f"\n--- ANÁLISIS DE RESULTADOS ---")
    print(f"1. OBSOLESCENCIA: Los vehículos pre-2000 tienen un índice de ({viejos:.4f}).")
    print(f"2. TECNOLOGÍA: Los vehículos modernos (2011-2020) marcan ({modernos:.4f}).")
    print(f"3. CONCLUSIÓN: La diferencia entre estos índices representa cuántas vidas")
    print(f"   ha salvado la evolución de la seguridad automotriz en California.")
except:
    print("\n--- ANÁLISIS DE RESULTADOS ---")
    print("Nota: No se pudieron comparar rangos específicos, revise la tabla superior.")

   EXPERIMENTO #4: SEGURIDAD VEHICULAR (AÑO DEL CARRO VS. MUERTE)

shape: (5, 4)
┌────────────────────────────┬──────────────────┬───────────────┬───────────────────┐
│ era_vehiculo               ┆ Total_Accidentes ┆ Total_Muertos ┆ Indice_Mortalidad │
│ ---                        ┆ ---              ┆ ---           ┆ ---               │
│ str                        ┆ u32              ┆ i64           ┆ f64               │
╞════════════════════════════╪══════════════════╪═══════════════╪═══════════════════╡
│ 1. Clásicos/Viejos (<2000) ┆ 270048           ┆ 2718          ┆ 0.010065          │
│ 2. Década 2000 (2001-2010) ┆ 853281           ┆ 7202          ┆ 0.00844           │
│ 3. Modernos (2011-2020)    ┆ 1238056          ┆ 7587          ┆ 0.006128          │
│ 4. Último Modelo (>2020)   ┆ 6804             ┆ 40            ┆ 0.005879          │
│ 5. Desconocido/Otros       ┆ 205362           ┆ 696           ┆ 0.003389          │
└────────────────────────────┴──────────────────┴──────────

In [61]:
import polars as pl

# 1. Cargamos datos
query_col = "SELECT case_id, killed_victims FROM collisions"
query_par = "SELECT case_id, cellphone_in_use FROM parties WHERE party_type = 'driver'"

df_col = pl.read_database_uri(query=query_col, uri=uri, engine="adbc")
df_par = pl.read_database_uri(query=query_par, uri=uri, engine="adbc")

# 2. LIMPIEZA CRÍTICA: Eliminar duplicados de case_id en parties
# Así evitamos que un accidente con 3 personas duplique las muertes por 3.
df_par_unique = df_par.unique(subset=["case_id"])

# 3. Join y Clasificación
df_final = df_col.join(df_par_unique, on="case_id", how="inner").with_columns([
    pl.col("cellphone_in_use").cast(pl.String).fill_null("0"),
    pl.col("killed_victims").cast(pl.Int64).fill_null(0)
])

# Definimos quién usó celular (basado en el 1 que vimos en tu tabla bruta)
df_analisis = df_final.with_columns(
    pl.when(pl.col("cellphone_in_use") == "1")
    .then(pl.lit("Uso de Celular"))
    .otherwise(pl.lit("Sin Uso/No Reportado"))
    .alias("categoria")
)

# 4. Cálculo Real
reporte_celular = (
    df_analisis.group_by("categoria")
    .agg([
        pl.len().alias("Total_Accidentes"),
        pl.col("killed_victims").sum().alias("Total_Muertos")
    ])
    .with_columns(
        (pl.col("Total_Muertos") / pl.col("Total_Accidentes")).alias("Indice_Mortalidad")
    )
    .sort("Indice_Mortalidad", descending=True)
)

# 5. INTERPRETACIÓN DE LA VERDAD
print("================================================================")
print("   EXPERIMENTO #5 (VERSIÓN FINAL): EL CELULAR SIN DUPLICADOS")
print("================================================================\n")
print(reporte_celular)

try:
    con_cel = reporte_celular.filter(pl.col("categoria") == "Uso de Celular")["Indice_Mortalidad"][0]
    total_muertos = reporte_celular["Total_Muertos"].sum()
    
    print(f"\n--- ANÁLISIS DE RESULTADOS ---")
    print(f"1. TOTAL REAL: Ahora el total de muertes es de {total_muertos},")
    print(f"   lo cual sí es coherente con los registros estatales.")
    print(f"2. LETALIDAD: El uso del celular registra un índice de ({con_cel:.4f}).")
    print(f"3. CONCLUSIÓN: El celular no solo causa más choques, sino que")
    print(f"   al ocurrir a altas velocidades por falta de reacción, la letalidad se mantiene alta.")
except:
    print("\nError en el cálculo de índices. Verifique la tabla superior.")

   EXPERIMENTO #5 (VERSIÓN FINAL): EL CELULAR SIN DUPLICADOS

shape: (2, 4)
┌──────────────────────┬──────────────────┬───────────────┬───────────────────┐
│ categoria            ┆ Total_Accidentes ┆ Total_Muertos ┆ Indice_Mortalidad │
│ ---                  ┆ ---              ┆ ---           ┆ ---               │
│ str                  ┆ u32              ┆ i64           ┆ f64               │
╞══════════════════════╪══════════════════╪═══════════════╪═══════════════════╡
│ Sin Uso/No Reportado ┆ 1419343          ┆ 11671         ┆ 0.008223          │
│ Uso de Celular       ┆ 25140            ┆ 197           ┆ 0.007836          │
└──────────────────────┴──────────────────┴───────────────┴───────────────────┘

--- ANÁLISIS DE RESULTADOS ---
1. TOTAL REAL: Ahora el total de muertes es de 11868,
   lo cual sí es coherente con los registros estatales.
2. LETALIDAD: El uso del celular registra un índice de (0.0078).
3. CONCLUSIÓN: El celular no solo causa más choques, sino que
   al ocurrir a

In [69]:
#================================================================
# CONCLUSIONES FINALES: RADIOGRAFÍA DE LA FATALIDAD EN CALIFORNIA
#================================================================

def imprimir_conclusiones():
    print("==================================================================")
    print("   RESUMEN EJECUTIVO: LOS 5 PILARES DE LA SEGURIDAD VIAL")
    print("==================================================================\n")
    
    conclusiones = [
        ("1. COMPORTAMIENTO", "El cinturón triplica la supervivencia. No usarlo es el factor individual de riesgo más alto."),
        ("2. TEMPORALIDAD", "La 1:00 AM es la 'Hora Crítica'. Menos tráfico, pero mayor letalidad por fatiga y visibilidad + gente tomando alcohol"),
        ("3. VULNERABILIDAD", "El peatón es el eslabón más débil. El índice 0.072 es la mayor sentencia de muerte."),
        ("4. TECNOLOGÍA", "La ingeniería salva vidas. Los autos modernos (2011+) reducen la mortalidad casi a la mitad vs. modelos pre-2000."),
        ("5. DISTRACCIÓN", "El celular es un multiplicador de accidentes. Su letalidad es baja por impacto, pero masiva por volumen.")
    ]
    
    for titulo, desc in conclusiones:
        print(f"● {titulo:18} | {desc}")

    print("\n" + "="*66)
    print(" REFLEXIÓN FINAL SOBRE EL CELULAR vs. VELOCIDAD:")
    print("="*66)
    print("Aunque el índice de letalidad del celular (0.0078) es menor al promedio general (0.0082),")
    print("esto revela una 'Falsa Seguridad'. El celular causa el ACCIDENTE (distracción),")
    print("mientras que la VELOCIDAD y el ALCOHOL definen la MUERTE (fuerza bruta).")
    print("El celular envía personas al hospital pero la física del impacto las envía a la tumba XD")
    print("="*66)

# Ejecutamos la impresión
imprimir_conclusiones()

   RESUMEN EJECUTIVO: LOS 5 PILARES DE LA SEGURIDAD VIAL

● 1. COMPORTAMIENTO  | El cinturón triplica la supervivencia. No usarlo es el factor individual de riesgo más alto.
● 2. TEMPORALIDAD    | La 1:00 AM es la 'Hora Crítica'. Menos tráfico, pero mayor letalidad por fatiga y visibilidad + gente tomando alcohol
● 3. VULNERABILIDAD  | El peatón es el eslabón más débil. El índice 0.072 es la mayor sentencia de muerte.
● 4. TECNOLOGÍA      | La ingeniería salva vidas. Los autos modernos (2011+) reducen la mortalidad casi a la mitad vs. modelos pre-2000.
● 5. DISTRACCIÓN     | El celular es un multiplicador de accidentes. Su letalidad es baja por impacto, pero masiva por volumen.

 REFLEXIÓN FINAL SOBRE EL CELULAR vs. VELOCIDAD:
Aunque el índice de letalidad del celular (0.0078) es menor al promedio general (0.0082),
esto revela una 'Falsa Seguridad'. El celular causa el ACCIDENTE (distracción),
mientras que la VELOCIDAD y el ALCOHOL definen la MUERTE (fuerza bruta).
El celular envía per